# Bespoke Radar

In [1]:
import io
import time
import random
import json
import re
import pandas as pd
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
import google.genai as genai
from google.genai import types
from google.cloud import storage, bigquery, secretmanager
from google.api_core import exceptions

# Configuration

In [2]:
PROJECT_ID = 'converged-brandradar-poc'
DATASET_ID = "brandradar"
REGION = 'europe-west2'
SECRET_ID = "GEMINI_API_KEY"

# Model and Parameters
# MODELNAME = 'models/gemini-3-pro-preview'
MODELNAME = 'models/gemini-3.1-pro-preview'
# MODELNAME = 'models/gemini-3.1-flash-lite'
# MODELNAME = 'models/gemini-flash-lite-latest'
# MODELNAME = 'models/gemini-2.5-pro'

BESPOKE = True
REPORTNAME = 'DESIRE Bespoke 3'
DATASOURCE = 'gs://converged-brandradar/lookups/Socialbespoke3.csv'

# Templates
TEMPLATES = {
    'Bespoke': 'gs://converged-brandradar/templates/Bespoke_Template.xlsx'
}

# Utilities

In [3]:
def access_secret_version(project_id: str, secret_id: str, version_id: str = "latest") -> str:
    """Accesses the payload for the given secret version."""
    client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    try:
        response = client.access_secret_version(request={"name": name})
        return response.payload.data.decode("UTF-8")
    except Exception as e:
        print(f"Error accessing secret: {e}")
        return None


In [4]:
def extract_json_robust(response_text):
    """Robust JSON extraction with multiple fallback strategies."""
    if not response_text:
        return None

    for pattern in [r'```json\s*(\{.*?\})\s*```', r'(\{.*\})']:
        match = re.search(pattern, response_text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1))
            except json.JSONDecodeError:
                pass

    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        return None

In [5]:
def retry_with_backoff(retries=5, backoff_in_seconds=1):
    def decorator(f):
        def wrapper(*args, **kwargs):
            _retries, _backoff = retries, backoff_in_seconds
            while _retries > 1:
                try:
                    return f(*args, **kwargs)
                except exceptions.ResourceExhausted as e:
                    sleep = _backoff + random.uniform(0, 1)
                    print(f"API quota exceeded. Retrying in {sleep:.2f}s...")
                    time.sleep(sleep)
                    _backoff *= 2
                    _retries -= 1
                except Exception as e:
                    print(f"An unexpected error occurred: {e}. Retrying...")
                    time.sleep(_backoff)
                    _retries -= 1
            return f(*args, **kwargs)
        return wrapper
    return decorator

In [6]:
def get_question(brand, category, market):
        return f"""Could you answer the following questions with a value in the range 0-100, if not a pewrcentage then this score should be relative to other brands in the category, and a short rationale in approximately 100 words each and
        return the answers in a JSON object with the variables called the question numbers, for example Q11, and the descriptions called, for example, Q11Description

        Q1:  What is the volume of {brand} mentions compared to its competitors in recent conversations? Provide a relative score or ranking to other brands.
        Q2:  What percentage of {brand} mentions in recent conversations is positive? Provide a relative score in comparison to competitors or ranking to other brands.
        Q3:  What percentage of {brand} mentions in recent conversations is negative? Provide a relative score in comparison to competitors or ranking to other brands
        Q4:  How many shares does each mention of {brand} receive in comparison to its competitors? Provide a relative score or ranking to other brands.
        Q5:  How much engagement did each mention of {brand} receive in comparison to its competitors in the past year? Provide a relative score or ranking to other brands.
        Q6:  How many shares did each mention of {brand} receive in comparison to its competitors in the past year? Provide a relative score or ranking to other brands.
        Q7:  What share of the market did {brand} have in the {market}



         """

# State and Schema Definitions

In [7]:
ANALYSIS_CONFIGS = {
    'Bespoke': {
        'run': BESPOKE,
        'sheet': 'Bespoke Results',
        'template': TEMPLATES['Bespoke'],
        'int_cols': ["Q1", "Q2", "Q3","Q4","Q5","Q6", "Q7"],
        # , "Q8","Q9","Q10","Q11", "Q12", "Q13","Q14","Q15","Q16","Q17"],
        'str_cols': ["Q1Description", "Q2Description", "Q3Description", "Q4Description", "Q5Description", "Q6Description", "Q7Description"]
        # ,
        #              "Q8Description", "Q9Description", "Q10Description", "Q11Description", "Q12Description", "Q13Description", "Q14Description",
        #              "Q15Description","Q16Description", "Q17Description"]
    }
}

# Attach full columns list for dataframe initialization mapping
for t, c in ANALYSIS_CONFIGS.items():
    c['columns'] = ["Brand", "Category", "Market"] + \
    c['int_cols'] + \
    c['str_cols'] + \
    ["date"]

In [8]:
def initialize_dataframe(task_name):
    """Dynamically set up DataFrame based on schema."""
    cols = ANALYSIS_CONFIGS[task_name]['columns']
    df = pd.DataFrame(columns=cols)
    for col in ANALYSIS_CONFIGS[task_name]['int_cols']:
        df[col] = pd.Series(dtype='int64')
    df['date'] = pd.Series(dtype='datetime64[ns]')
    return df


# API Logic

In [9]:
@retry_with_backoff(retries=3, backoff_in_seconds=2)
def run_query(client, question):
    try:
        response = client.models.generate_content(
        model=MODELNAME,
        contents=question,
        config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level="LOW"),
        temperature=1.0,
        tools=[types.Tool(google_search=types.GoogleSearch())]
        )
        )
        if not response.candidates:
            print(f"Request blocked. Feedback: {response.prompt_feedback}")
            return None
        return response.text
    except ValueError:
        print("Response empty or blocked.")
        return None




In [10]:
def extract_values_to_row(task_name, json_data, row_template):
    """Parses logic correctly depending on specific task expectations."""
    if not json_data:
        return row_template

    config = ANALYSIS_CONFIGS[task_name]
    new_row = row_template.copy()

    # Generic mappings
    if task_name != 'Culture':
        new_row['Summary'] = json_data.get('summary', '')
    #  new_row['Summary'] = json_data.get('summary', '')

    if task_name == 'Social':
        scores = json_data.get('scores', {})
        for measure in config['int_cols']:
            # Adjust the key naming difference specific for social
            key = "sentiment_shift" if measure == "shift" else "sentiment_spike" if measure == "spike" else measure
            if key in scores:
                new_row[measure] = scores[key].get('score', 0)
                new_row[f"{measure}_description"] = scores[key].get('description', '')
    else:
        for k, v in json_data.items():
            if k in new_row or k not in config['columns']: continue
            new_row[k] = v

    return new_row


In [11]:
def process_record(client, row, task_name):
    """Processes a single brand query."""
    question = get_question(row['brand'], row['category'], row['market'])
    response_text = run_query(client, question)

    brand_json = extract_json_robust(response_text) if response_text else None

    # Init base row structure
    row_template = {
        "Brand": brand_json.get('brand', row['brand']) if brand_json else row['brand'],
        "Category": row['category'],
        "Market": row['market'],
        "date": datetime.now().strftime("%Y/%m/%d")
    }
    if task_name == 'Culture':
        row_template["projectname"] = REPORTNAME

    row_data = extract_values_to_row(task_name, brand_json, row_template)

    return row_data, question, response_text, bool(brand_json)


In [12]:
def execute_task(task_name, df_input, client, storage_client):
    """Executes the full pipeline for a specific reporting mode."""
    print(f"\n{'='*40}\nProcessing Task: {task_name}\n{'='*40}")

    df = initialize_dataframe(task_name)
    all_responses = []

    for idx, row in df_input.iterrows():
        print(f"[{idx+1}/{len(df_input)}] Processing {row['brand']} ({row['market']})")
        row_data, question, resp_text, success = process_record(client, row, task_name)

        df = pd.concat([df, pd.DataFrame([row_data])], ignore_index=True)

        all_responses.append({
            "timestamp": datetime.now().isoformat(),
            "brand": row['brand'],
            "market": row['market'],
            "input_question": question,
            "gemini_response": resp_text,
            "success": success
        })
        time.sleep(1) # Rate limiter
        # print(resp_text)

    df['projectname'] = REPORTNAME

    # -----------------------------------------

    # Exporters
    bucket = storage_client.bucket("converged-brandradar")

    # 1. JSON
    blob_name = f"results/{task_name} {datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    bucket.blob(blob_name).upload_from_string(json.dumps(all_responses, indent=4), content_type='application/json')
    print(f"JSON Exported: gs://converged-brandradar/{blob_name}")

    # 2. XLS
    config = ANALYSIS_CONFIGS[task_name]
    xls_path = f"gs://converged-brandradar/results/{task_name} {REPORTNAME.strip()}.xlsx"

    template_bucket = storage_client.bucket(config['template'].split('/')[2])
    template_blob = template_bucket.blob('/'.join(config['template'].split('/')[3:]))
    template_buffer = io.BytesIO()
    template_blob.download_to_file(template_buffer)
    template_buffer.seek(0)

    book = load_workbook(template_buffer)
    book['Key'].cell(row=3, column=1, value=REPORTNAME)
    sheet = book[config['sheet']]

    for r_idx, out_row in enumerate(dataframe_to_rows(df, index=False, header=False), 2):
        for c_idx, value in enumerate(out_row, 1):
            sheet.cell(row=r_idx, column=c_idx, value=value)

    out_buffer = io.BytesIO()
    book.save(out_buffer)
    out_buffer.seek(0)

    out_bucket = storage_client.bucket(xls_path.split('/')[2])
    out_bucket.blob('/'.join(xls_path.split('/')[3:])).upload_from_file(out_buffer, content_type='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')
    print(f"Excel Exported: {xls_path}")




# Entrypoint

In [13]:
def main():
    start_time = time.time()

    api_key = access_secret_version(PROJECT_ID, SECRET_ID)
    if not api_key:
        print("Failed to retrieve API key!")
        return

    client = genai.Client(api_key=api_key)
    storage_client = storage.Client()

    # Load and process Data
    dfbrands = pd.read_csv(DATASOURCE)
    # dfbrands = dfbrands.head(1) # Drop head() modifier when ready
    # dfbrands = dfbrands.iloc[5:] # Drop a number of records if rerunning or splitting
    cats = dfbrands.groupby(['market', 'category']).size().reset_index()
    cats['brand'] = 'All Category'
    dfbrands = pd.concat([dfbrands, cats], ignore_index=True)

    for task_name, config in ANALYSIS_CONFIGS.items():
        if config['run']:
            execute_task(task_name, dfbrands, client, storage_client)

    print(f"\nTotal Run time: {time.time() - start_time:.2f} seconds")
    # Remove Category totals for Culture
    # for m in client.models.list():
    #     print(m.name)



In [14]:
if __name__ == "__main__":
    main()


Processing Task: Bespoke
[1/1118] Processing Volkswagen (India)
[2/1118] Processing Bridgestone (India)
[3/1118] Processing Ceat (India)
[4/1118] Processing CEAT Tyres (India)
[5/1118] Processing Goodyear (India)
[6/1118] Processing Michelin (India)
[7/1118] Processing MRF Tyres (India)
[8/1118] Processing TVS Eurogrip (India)
[9/1118] Processing TVS Tyres (India)
[10/1118] Processing Dove (India)
[11/1118] Processing Dove Skin Care (India)
[12/1118] Processing Garnier (India)
[13/1118] Processing Himalaya (India)
[14/1118] Processing Lakme (India)
[15/1118] Processing L'Or√©al Paris (India)
[16/1118] Processing L'Or√©al Paris Make Up (India)
[17/1118] Processing Mama Earth (India)
[18/1118] Processing Olay (India)
[19/1118] Processing POND'S (India)
[20/1118] Processing Amazon Prime Video (India)
[21/1118] Processing Disney+ (India)
[22/1118] Processing Gaana (India)
[23/1118] Processing Lionsgate Play (India)
[24/1118] Processing Netflix (India)
[25/1118] Processing Smule (India)
[2

/tmp/ipykernel_175/2915677372.py:12: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([row_data])], ignore_index=True)


[362/1118] Processing Adidas (Spain)
[363/1118] Processing Asics (Spain)
[364/1118] Processing Calzedonia (Spain)
[365/1118] Processing Converse (Spain)
[366/1118] Processing Ecoalf (Spain)
[367/1118] Processing El Corte Ingl√©s (Spain)
[368/1118] Processing Geox (Spain)
[369/1118] Processing H&M (Spain)
[370/1118] Processing Intimissimi (Spain)
[371/1118] Processing Mango (Spain)
[372/1118] Processing Moda El Corte Ingl√©s (Spain)
[373/1118] Processing Nike (Spain)
[374/1118] Processing Primark (Spain)
[375/1118] Processing Puma (Spain)
[376/1118] Processing Reebok (Spain)
[377/1118] Processing Zara (Spain)
[378/1118] Processing Citro√´n (Spain)
[379/1118] Processing Cupra (Spain)
[380/1118] Processing Dacia (Spain)
[381/1118] Processing Hyundai (Spain)
[382/1118] Processing Jeep (Spain)
[383/1118] Processing Kia (Spain)
[384/1118] Processing Nissan (Spain)
[385/1118] Processing Opel (Spain)
[386/1118] Processing Peugeot (Spain)
[387/1118] Processing Renault (Spain)
[388/1118] Process

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/aqa
models/imagen-4.0-generate-001
models/imagen-4.0-ultra-generate-001
models/imagen-4.0-fast-generate-001
models/veo-2.0-generate-001
models/veo-3.0-generate-001
models/veo-3.0-fast-generate-001
models/veo-3.1-generate-preview
models/veo-3.1-fast-generate-preview
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
